# 📊 Government ABAC Demo - Step 3: Setup Governed Tags

## 📋 Overview
This notebook creates the tag policies defining the necessary governed tags and their allowed values

### What This Notebook Does:
1. **Creates Tag policy**: Uses REST API to create tag policies defining governed tags and their allowed values
2. **Demonstrates Usage**: Shows how to apply tags to relevant columns in the government datasets

## 🎓 How to Use This Notebook
1. **Ensure Steps 1-2 Complete**: All functions, tables, and data must exist
2. **Run All Cells**: Execute sequentially to see all test results
3. **Verify Expectations**: Check that tags have been applied through the catalog explorer

## ⚙️ Prerequisites
- ✅ **Step 1 completed**: All functions created
- ✅ **Step 2 completed**: Core tables with data
- ✅ APPLY TAG permission on all tables

## 📊 Expected Results
After running this notebook:
- ✅ Tag policies are created
- ✅ Governed tags assigned
- ✅ Prepared for creating ABAC policies

## ⚙️ Configuration

Using the same catalog and schema from previous steps:
- **Catalog**: `your_catalog_name`
- **Schema**: `government`


In [0]:
pip install pyyaml

In [0]:
# 📋 Load Configuration from config.yaml
import yaml
from pathlib import Path
import requests
import os
from databricks.sdk import WorkspaceClient

config_file = Path('config.yaml')
if config_file.exists():
    with open(config_file) as f:
        config = yaml.safe_load(f)
    CATALOG = config['catalog']
    SCHEMA = config['schema']
    print(f'✅ Configuration loaded from config.yaml')
    print(f'   📊 Catalog: {CATALOG}')
    print(f'   📁 Schema: {SCHEMA}')
else:
    # Fallback defaults
    CATALOG = 'your_catalog_name'
    SCHEMA = 'government'
    print(f'⚠️  config.yaml not found - using defaults')
    print(f'   📊 Catalog: {CATALOG}')
    print(f'   📁 Schema: {SCHEMA}')

# Set catalog and schema to use for the cells below
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")

client = WorkspaceClient()
workspace_url = client.config.host

In [0]:
%sql
SELECT '🎯 Target: ' || current_catalog() || '.' || current_schema() AS status;

###Define Governed Tags + Allowed Values
https://docs.databricks.com/api/workspace/tagpolicies/createtagpolicy

In [0]:
def get_token():
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    return getattr(ctx, "apiToken")().get()


def create_tag_policy(payload):
    data = requests.post(
        f"{workspace_url}/api/2.1/tag-policies",
        headers={"Authorization": f"Bearer {get_token()}"},
        json=payload,
    )

    return data

In [0]:
pii_payload = {
    "description": "Government PII data types",
    "tag_key": "pii_type_government",
    "values": [
        {"name": "ssn"},
        {"name": "name"},
        {"name": "dob"},
        {"name": "address"},
        {"name": "phone"},
        {"name": "email"},
        {"name": "id"},
        {"name": "amount"},
        {"name": "license"}
    ]
}
print(create_tag_policy(pii_payload).json())

security_classification_payload = {
    "description": "Security classification levels",
    "tag_key": "security_classification_government",
    "values": [
        {"name": "Top_Secret"},
        {"name": "Secret"},
        {"name": "Confidential"},
        {"name": "Unclassified"}
    ]
}
print(create_tag_policy(security_classification_payload).json())

data_sensitivity_payload = {
    "description": "Data sensitivity levels",
    "tag_key": "data_sensitivity_government",
    "values": [
        {"name": "CUI"},
        {"name": "FOUO"},
        {"name": "Sensitive"},
        {"name": "Public"}
    ]
}
print(create_tag_policy(data_sensitivity_payload).json())

In [0]:
%sql
ALTER TABLE citizens ALTER COLUMN citizen_id SET TAGS ('pii_type_government' = 'id', 'security_classification_government' = 'Secret');
ALTER TABLE citizens ALTER COLUMN first_name SET TAGS ('pii_type_government' = 'name');
ALTER TABLE citizens ALTER COLUMN last_name SET TAGS ('pii_type_government' = 'name');
ALTER TABLE citizens ALTER COLUMN ssn SET TAGS ('pii_type_government' = 'ssn', 'data_sensitivity_government' = 'Sensitive');
ALTER TABLE citizens ALTER COLUMN address SET TAGS ('pii_type_government' = 'address');

In [0]:
%sql
ALTER TABLE licenses ALTER COLUMN citizen_id SET TAGS ('pii_type_government' = 'id', 'security_classification_government' = 'Secret');
ALTER TABLE licenses ALTER COLUMN license_number SET TAGS ('pii_type_government' = 'license', 'data_sensitivity_government' = 'Sensitive');

In [0]:
%sql
ALTER TABLE tax_records ALTER COLUMN tax_owed SET TAGS ('pii_type_government' = 'amount');
ALTER TABLE tax_records ALTER COLUMN citizen_id SET TAGS ('pii_type_government' = 'id', 'security_classification_government' = 'Secret');

In [0]:
%sql
ALTER TABLE violations ALTER COLUMN fine SET TAGS ('data_sensitivity_government' = 'Sensitive');

In [0]:
%sql
SELECT '✅ Tags applied successfully to government tables!' AS status;

## ✅ Success!

Tag policies have been created successfully and governed tags have been assigned!

### What You Just Created:
- ✅ Governed tags for capturing data sensitivity 
- ✅ Tag assignment to tables

### 🎯 Next Step:

Continue to **`4_Test_ABAC_Policies.ipynb`** to define ABAC policies using governed tags and test them on the datasets in the government schema

---